# CS 231N — Sign Language Translation
## Notebook 1 : Preprocessing + Entraînement
**Marie Imad & Aaron Sequeira — Stanford 2026**

Ce notebook suit le pipeline complet :
1. Setup
2. Preprocessing des vidéos (frames + keypoints)
3. Vérification du dataset
4. Entraînement
5. Courbes d'apprentissage

In [ ]:
# ── 1. Mount Drive + cloner le repo ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/cs231n_sign'
CODE = '/content/sign_language'

# Option A : upload les .py depuis ton ordi
# Option B : clone depuis GitHub (recommandé)
# !git clone https://github.com/TON_REPO/sign_language.git {CODE}

import sys
sys.path.insert(0, CODE)
print('✅ Code path ajouté')

In [ ]:
# ── 2. Install deps ────────────────────────────────────────────
!pip install -q mediapipe transformers==4.40.0 evaluate sacrebleu
!pip install -q av
print('✅ Deps installées')

In [ ]:
# ── 3. Preprocessing ───────────────────────────────────────────
# À lancer UNE SEULE FOIS. Ensuite les .npy sont dans Drive.

from preprocessing import preprocess_dataset
import os

for split in ('train', 'val', 'test'):
    preprocess_dataset(
        annotation_csv = f'{ROOT}/annotations/{split}.csv',
        videos_dir     = f'{ROOT}/raw',
        frames_out     = f'{ROOT}/frames',
        kp_out         = f'{ROOT}/keypoints',
        split          = split,
        # max_videos   = 50,  # décommente pour tester sur 50 clips
    )

In [ ]:
# ── 4. Vérifier un sample ──────────────────────────────────────
from dataset import How2SignDataset
import torch

ds = How2SignDataset(
    annotation_csv = f'{ROOT}/annotations/train.csv',
    frames_dir     = f'{ROOT}/frames',
    keypoints_dir  = f'{ROOT}/keypoints',
    split          = 'train',
)
sample = ds[0]
print('frames    :', sample['frames'].shape)      # (48, 3, 224, 224)
print('keypoints :', sample['keypoints'].shape)   # (48, 225)
print('labels    :', sample['labels'].shape)      # (50,)
print('caption   :', sample['caption'])

In [ ]:
# ── 5. Test du modèle (forward pass) ──────────────────────────
from model import SignLanguageTranslator, count_params

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

model = SignLanguageTranslator().to(device)
count_params(model)

# Mini batch de test
frames    = sample['frames'].unsqueeze(0).to(device)     # (1, 48, 3, 224, 224)
keypoints = sample['keypoints'].unsqueeze(0).to(device)  # (1, 48, 225)
labels    = sample['labels'].unsqueeze(0).to(device)     # (1, 50)
att_mask  = sample['attention_mask'].unsqueeze(0).to(device)

out = model(frames, keypoints, labels, att_mask)
print(f'\n✅ Forward pass OK | Loss = {out.loss.item():.4f}')

In [ ]:
# ── 6. Lancer l'entraînement ───────────────────────────────────
from train import main

main(
    root           = ROOT,
    checkpoint_dir = f'{ROOT}/checkpoints',
)

In [ ]:
# ── 7. Courbes d'apprentissage ─────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f'{ROOT}/checkpoints/training_log.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(log['epoch'], log['val_bleu'], color='green')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('BLEU')
axes[1].set_title('BLEU Score (Val)')

plt.tight_layout()
plt.savefig(f'{ROOT}/checkpoints/training_curves.png', dpi=150)
plt.show()
print('✅ Courbes sauvegardées')